# Dependency and Supply Chain Protection - End-to-End Pipeline
**Date**: 2026-05-31  
**Objective**: Estimate release risk from dependency and artifact integrity gaps.


## 1. Imports and Setup


In [ ]:
from __future__ import annotations

import os
from pathlib import Path
import sys

import pandas as pd

module_dir = Path('vulnerability-defenses').resolve()
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from shared_vulnerability_notebook_lib import (
    DefenseConfig,
    TOPIC_PROFILES,
    build_dataset,
    build_summary,
    evaluate,
    parse_use_gpu_flag,
    persist_metrics,
    plot_metric_bars,
    plot_score_hist,
    predict_scores,
    prepare_features,
    quick_eda,
    resolve_runtime_device,
    set_reproducibility,
    train_logistic,
)

SEED = 42
set_reproducibility(SEED)
USE_GPU = parse_use_gpu_flag(os.getenv('USE_GPU', '1'))
runtime_device = resolve_runtime_device(USE_GPU)
print(f'USE_GPU={int(USE_GPU)} | runtime_device={runtime_device}')
            


## 2. Configuration and Constants


In [ ]:
CFG = DefenseConfig(
    topic_key='dependency-supply-chain-protection',
    topic_title='Dependency and Supply Chain Protection',
    seed=SEED,
    test_ratio=0.25,
    threshold=0.57,
    output_dir=Path('vulnerability-defenses/outputs'),
)
PROFILE = TOPIC_PROFILES[CFG.topic_key]
CFG
            


## 3. Data Loading


In [ ]:
df = build_dataset(CFG, PROFILE, n_samples=260)
print(f'rows={len(df)} cols={len(df.columns)}')
df.head()
            


## 4. Exploratory Data Analysis (EDA)


In [ ]:
eda = quick_eda(df)
print('incident_rate:', round(eda['incident_rate'], 4))
pd.Series(eda['feature_mean']).sort_values(ascending=False)
            


## 5. Preprocessing / Feature Engineering


In [ ]:
x_train, y_train, x_test, y_test = prepare_features(
    df=df,
    test_ratio=CFG.test_ratio,
    seed=CFG.seed,
)
print(
    f'x_train={x_train.shape}, y_train={y_train.shape}, '
    f'x_test={x_test.shape}, y_test={y_test.shape}'
)
            


## 6. Model Definition


In [ ]:
EPOCHS = 320
LEARNING_RATE = 0.11
print(f'epochs={EPOCHS} learning_rate={LEARNING_RATE}')
            


## 7. Training


In [ ]:
w, b, losses = train_logistic(
    x_train=x_train,
    y_train=y_train,
    epochs=EPOCHS,
    lr=LEARNING_RATE,
)
print(f'final_loss={losses[-1]:.4f}')
            


## 8. Evaluation and Metrics


In [ ]:
test_scores = predict_scores(x_test, w, b)
metrics = evaluate(y_true=y_test, scores=test_scores, threshold=0.5)
metrics
            


## 9. Results Visualization


In [ ]:
plot_metric_bars(metrics, 'Dependency and Supply Chain Protection - Metric Summary')
plot_score_hist(test_scores, y_test, 'Dependency and Supply Chain Protection - Score Distribution')
metrics_path = persist_metrics(CFG, metrics)
print(f'metrics saved to: {metrics_path}')
            


## 10. Summary / Conclusions


In [ ]:
summary = build_summary(CFG.topic_title, metrics, runtime_device)
print(summary)
print('Use this baseline to tune thresholds and rollout controls per environment.')
            
